## Chatbot Config

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
from langchain_groq import ChatGroq
groq = ChatGroq(model="openai/gpt-oss-120b", api_key= os.getenv("GROQ_API_KEY"))

c:\Users\singh\OneDrive\Desktop\Generative AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.messages import HumanMessage
groq.invoke([HumanMessage(content="Hi, My name is Navneet and I am a AI Engineer")])

AIMessage(content='Hello Navneet! It’s great to meet an AI Engineer. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is Navneet and I am a AI Engineer". Likely they are greeting. We should respond politely, maybe ask how we can help. No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 84, 'total_tokens': 154, 'completion_time': 0.151135604, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.003386188, 'prompt_tokens_details': None, 'queue_time': 0.046111492, 'total_time': 0.154521792}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bc9a6c9b5f', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab59-b83a-7170-bc83-912c8a2b8771-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 70, 'total_tokens': 154, 'output_token_details': {'reasoning': 41}})

In [4]:
from langchain_core.messages import AIMessage
groq.invoke(
    [
        HumanMessage(content="Hi, My name is Navneet and I am a AI Engineer"),
        AIMessage(content="Hello Navneet! 👋 Great to meet an AI Engineer. How can I assist you today?"),
        HumanMessage(content="Hey Whats my name, and what do I do?")
    ]
)

AIMessage(content='Your name is **Navneet**, and you work as an **AI Engineer**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "Hey Whats my name, and what do I do?" The user previously said: "Hi, My name is Navneet and I am a AI Engineer". So answer: Name is Navneet, you are an AI Engineer. Probably also ask if they need anything else.'}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 125, 'total_tokens': 218, 'completion_time': 0.194323073, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.026166145, 'prompt_tokens_details': None, 'queue_time': 0.168296815, 'total_time': 0.220489218}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca17e3cea8', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab59-ba33-7c51-9927-4ed24928a245-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_token

## Message History

We can use a Message History class to wrap our model and make it stateful. Tgis will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of input.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(groq, get_session_history)

In [6]:
config = {"configurable" : {"session_id" : "chat1"}}

In [7]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Navneet and I am a AI Engineer"),

    ], config=config
)
response.content

'Hello Navneet! 👋\n\nGreat to meet an AI Engineer. How’s everything going in the world of models, data, and experiments? Is there anything specific you’d like to discuss or dive into today—whether it’s a technical challenge, a new idea you’re exploring, or just a quick chat about the latest AI trends? Let me know!'

In [8]:
with_message_history.invoke(
    [
        HumanMessage(content="What is my name?"),

    ], config=config
)

AIMessage(content='Your name is Navneet.', additional_kwargs={'reasoning_content': 'The user asks: "What is my name?" The user previously said: "Hi, My name is Navneet and I am a AI Engineer". So answer: Navneet.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 171, 'total_tokens': 223, 'completion_time': 0.109524506, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.007329338, 'prompt_tokens_details': None, 'queue_time': 0.047412317, 'total_time': 0.116853844}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab59-bdf9-7df0-9d04-a24c499e445e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 52, 'total_tokens': 223, 'output_token_details': {'reasoning': 37}})

In [9]:
## Changing the Config
config1 = {"configurable" : {"session_id" : "chat2"}}
response = with_message_history.invoke(
    [
        HumanMessage(content="What is my name?"),

    ], config=config1
)
response.content

'I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

## Prompt Templates

It helps to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM.

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assisstant, answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | groq

In [13]:
chain.invoke({"messages" : [HumanMessage(content="Hi, My name is Khushi")]})

AIMessage(content='Hello, Khushi! 👋 Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT, greeting the user, perhaps ask how can help. The user says "Hi, My name is Khushi". We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 99, 'total_tokens': 163, 'completion_time': 0.13299353, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.00366621, 'prompt_tokens_details': None, 'queue_time': 0.158652635, 'total_time': 0.13665974}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_46d8af1c62', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab5a-80b7-7171-b8ed-b433ee5e5570-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 64, 'total_tokens': 163, 'output_token_details': {'reasoning': 36}})

In [14]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)
config2 = {"configurable" : {"session_id" : "chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Khushi")],
    config=config2
)
response.content

'Hello, Khushi! Nice to meet you. How can I assist you today?'

In [15]:
## Multiple Variable
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assisstant, answer all quations to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | groq

response = chain.invoke({"messages" : [HumanMessage(content="Hello, my name is Ubaid")], "language" : "Hindi"})
response.content

'नमस्ते उबैद! आपसे मिलकर खुशी हुई। मैं आपकी कैसे मदद कर सकता हूँ?'

In [20]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config3 = {"configurable" : {"session_id" : "chat4"}}

response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Devendra")], "language": "Hindi"},
    config=config3 
)
response.content

'नमस्ते देवेंद्र जी! आपसे मिलकर खुशी हुई। मैं आपकी किस प्रकार मदद कर सकता हूँ?'

## Managing the Chat History

In [24]:
from langchain_core.messages import SystemMessage, trim_messages

## trim_messages ---> Helps to reduce how many messages we're sending to the model.

trimmer = trim_messages(
    max_tokens= 45,
    strategy= "last",
    token_counter= len,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content= "You are a good assisstant"),
    HumanMessage(content= "Hi, I am Bob"),
    AIMessage(content= "Hii!!"),
    HumanMessage(content= "I like vanilla ice cream"),
    AIMessage(content= "Nice"),
    HumanMessage(content= "Whats 2 + 2"),
    AIMessage(content= "4"),
    HumanMessage(content= "Thanks"),
    AIMessage(content="No Problem!"),
    HumanMessage(content="Having Fun?"),
    AIMessage(content= "Yes!")
]

trimmer.invoke(messages)

[SystemMessage(content='You are a good assisstant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi, I am Bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hii!!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No Problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Having Fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes!', additional_k

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer) 
    | prompt
    | groq
)

response = chain.invoke(
    {"messages" : messages + [HumanMessage(content="What Ice cream do i like?")],
     "language" : "Hindi"} 
)

response.content

'आपने बताया था कि आपको **वैनिला आइसक्रीम** पसंद है।'

In [29]:
response = chain.invoke(
    {"messages" : messages + [HumanMessage(content="What math problem did I asked?")],
     "language" : "English"} 
)

response.content

'You asked **“What’s\u202f2\u202f+\u202f2?”** – a simple addition problem.'

In [34]:
## Lets wrap it in message history
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key = "messages"
)

config4 = {"configurable" : {"session_id" : "chat5"}}

response = with_message_history.invoke(
    {"messages": messages + [HumanMessage(content="What is my name")], "language": "English"},
    config=config4 
)
response.content

'Your name is Bob.'